In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
import torchvision #for computer vision
from torchvision.datasets import CIFAR10

In [4]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [5]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #kernel size=2, stride=2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) #flattening
        x = self.fc_layers(x)

        return x

In [7]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [8]:
epochs = 10
epoch_training_loss = []

for epoch in range(epochs):
    training_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()
        
    tl = training_loss/len(trainloader)
    epoch_training_loss.append(tl)

    print(f"epochs {epoch+1} losses = {tl}")

epochs 1 losses = 1.3603267040856355
epochs 2 losses = 0.9208507144542606
epochs 3 losses = 0.72815777067943
epochs 4 losses = 0.6010981820276021
epochs 5 losses = 0.4999517730206175
epochs 6 losses = 0.40131274244898113
epochs 7 losses = 0.3194876665063679
epochs 8 losses = 0.24668055413590978
epochs 9 losses = 0.19372675229635689
epochs 10 losses = 0.14642196772691538


In [9]:
total_vals = 0
correct = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        correct += (predicted == labels).sum().item()
        total_vals += labels.size(0)

print(f"Accuracy = {correct/total_vals *100}")

Accuracy = 75.49
